# Streaming monitor — cold start, drift-triggered retraining, and the audit trail

This notebook walks through **`scripts/run_streaming_monitor.py`**'s
actual pipeline cell by cell, using the same functions the CLI script
calls — nothing here is reimplemented. It's the Phase 9 companion to
`dynamic_orchestrator.ipynb`.

**The architecture, in one sentence:** a long-running streaming session
is not one giant `run_dynamic_loop` call — this notebook cold-starts
via one ordinary `run_dynamic_loop` call (identical to a Phase 8 run),
then for each incoming batch pushes it into the persisted context and
calls `run_dynamic_loop` again, bounded, so the planner decides what to
do with that ONE batch: nothing, score it only (`infer_only`), or
retrain (`retrain`) — reusing the exact same catalog-and-validator
mechanism (`orchestrator/agent_registry.py`,
`orchestrator/dynamic_loop.py`'s `validate_plan`/`execute_agent_step`)
Phase 8 already has. `monitor_drift` and `infer_batch` are
deterministic (no LLM); `retrain_decision` is the one new LLM agent,
and it never retrains anything itself — it only proposes an action the
harness independently validates and executes, the same "agents
propose, the harness decides" discipline as every other agent here.

This is explicitly **not** a new trust mechanism.

## 0. Setup — locate the repo, load `.env`, import everything

In [1]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "agentic_ml").exists():
            return candidate
    raise RuntimeError("Could not find repo root (looked for src/agentic_ml)")


REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))
print(f"Repo root: {REPO_ROOT}")

Repo root: /home/ubuntupc/Documents/Github/agentic-ai-project/agentic-ml-classification


In [2]:
def load_env_file(path: Path) -> None:
    if not path.exists():
        print(f"No .env file found at {path} — assuming environment variables are already set.")
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip())


load_env_file(REPO_ROOT / ".env")

In [3]:
from agentic_ml.cli_common import make_run_dir, make_tracer, make_transcript_writer, resolve_model_endpoint
from agentic_ml.harness.dataset import read_dataframe
from agentic_ml.harness.streaming import simulate_batches
from agentic_ml.model_client import ModelClient
from agentic_ml.orchestrator.agent_registry import list_agent_summaries
from agentic_ml.orchestrator.dynamic_loop import load_raw_hash, run_dynamic_loop
from agentic_ml.orchestrator.run_state import DynamicRunContext, RunStateSummary

## 1. The agent catalog, streaming-capable

`orchestrator/agent_registry.py` — the same fixed, auditable menu
`dynamic_orchestrator.ipynb` shows, plus three Phase 9 entries
(`monitor_drift`, `retrain_decision`, `infer_batch`), each gated behind
`requires_capability="streaming"`. An ordinary (non-streaming)
dynamic-orchestrator run never sees these three at all — capability is
only granted once `DynamicRunContext.accumulated_df` exists, which
happens the first time `finalize` runs (cold start here).

In [4]:
catalog_df = pd.DataFrame(list_agent_summaries(capabilities={"deep_dive", "streaming"}))
display(catalog_df[["agent_id", "title", "when_to_use", "required_state"]])

,agent_id,title,when_to_use,required_state
0,intake,Intake,"First step, whenever the target column isn't a...",{'target_known': False}
1,feature_engineering,Feature Engineering,"Immediately after the target is known, exactly...","{'target_known': True, 'feature_engineering_do..."
2,profiler,Profiler,After feature engineering has run (even if it ...,"{'target_known': True, 'feature_engineering_do..."
3,split_and_check_leakage,Split + Leakage Checks,"Immediately after the profiler, exactly once.","{'profiler_done': True, 'split_done': False}"
4,modeling,Modeling,After the split passes its leakage checks. Cal...,{'split_leakage_passed': True}
5,verification,Verification,Whenever a gate-passing candidate hasn't been ...,{'has_unverified_passing_candidate': True}
6,finalize,Finalize,Once at least one candidate has been verified ...,"{'has_verified_candidate': True, 'final_test_m..."
7,monitor_drift,Monitor Drift,Whenever a new batch has arrived and hasn't be...,"{'new_batch_pending': True, 'drift_checked': F..."
8,retrain_decision,Retrain Decision,Immediately after monitor_drift has run for th...,"{'drift_checked': True, 'pending_retrain_actio..."
9,infer_batch,Infer Batch,After retrain_decision has chosen infer_only f...,"{'pending_retrain_action': 'infer_only', 'batc..."


## 2. Configuration

Edit this cell for your dataset/endpoint, then run everything below it
in order. Defaults to the local server for fast, reliable notebook
execution (see README.md "Three model endpoints").

`n_initial_groups`/`batch_size_groups` are in units of `group_column`
(planes, here) — `harness/streaming.py::simulate_batches` never splits
one plane's flights across the cold-start/batch boundary. This
notebook subsets to the `__mean` sensor columns before replaying
batches, the same way `scripts/evaluate_streaming_monitor.py` does —
the full smoke file has 282 columns, wide enough that
`feature_engineering`'s tool output can overflow a 32K-context local
model's request budget (a pre-existing scalability limit of
`feature_engineering_step`/`profiler_step` on very wide tables, not
something Phase 9 introduces). Phase 9 also never runs
`feature_engineering` at all in a streaming session — see
`scripts/run_streaming_monitor.py`'s module docstring for why: every
batch is always a RAW slice of the source table, so the column schema
has to stay identical for the whole session.

In [5]:
CONFIG = {
    "data_path": "datasets/processed/ngafid_c28_flights_smoke.csv",
    "target_column": "before_after",
    "group_column": "plane_id",
    "id_column": "id",
    "id_columns": "id,date_diff,split",
    "strategy": "group",
    "n_initial_groups": 25,
    "batch_size_groups": 10,
    "max_batches": 2,          # keep this short for a notebook run
    "seed": 42,
    "drift_batch_index": 0,    # which batch gets a synthetic sensor-mean shift injected
    "metrics": "roc_auc,pr_auc,f1,accuracy",
    "max_iterations_cold_start": 15,
    "max_iterations_per_batch": 20,
    "model": None,
    "verification_model": None,
    "use_gateway": False,
    "use_local": True,
}

## 3. Build the replay dataset — subset columns, simulate batches, inject drift into one

In [6]:
NGAFID_MEAN_COLS_KEEP = None  # filled in below

full_df = pd.read_csv(CONFIG["data_path"])
keep_cols = [c for c in full_df.columns if c.endswith("__mean")] + [
    CONFIG["group_column"], CONFIG["id_column"], CONFIG["target_column"], "date_diff", "split",
]
full_df = full_df[[c for c in keep_cols if c in full_df.columns]]
NGAFID_MEAN_COLS_KEEP = [c for c in full_df.columns if c.endswith("__mean")]

initial_df, batches = simulate_batches(
    full_df, group_column=CONFIG["group_column"], id_column=CONFIG["id_column"],
    n_initial_groups=CONFIG["n_initial_groups"], batch_size_groups=CONFIG["batch_size_groups"],
    seed=CONFIG["seed"],
)
batches = batches[: CONFIG["max_batches"]]
print(f"Cold-start pool: {len(initial_df)} rows across {initial_df[CONFIG['group_column']].nunique()} planes.")
for i, b in enumerate(batches):
    print(f"  batch {i}: {len(b)} rows across {b[CONFIG['group_column']].nunique()} planes")

Cold-start pool: 49 rows across 25 planes.
  batch 0: 20 rows across 10 planes
  batch 1: 16 rows across 10 planes


In [7]:
# Inject a large synthetic shift on CONFIG["drift_batch_index"]'s planes, on every
# kept sensor mean — makes that ONE batch genuinely, obviously out-of-distribution
# relative to the cold-start pool, so retrain_decision has real evidence to act on.
rng = np.random.RandomState(CONFIG["seed"] + 1)
drift_batch_idx = CONFIG["drift_batch_index"]
drift_plane_ids = set(batches[drift_batch_idx][CONFIG["group_column"]])
mask = full_df[CONFIG["group_column"]].isin(drift_plane_ids)
for col in NGAFID_MEAN_COLS_KEEP:
    col_std = full_df[col].std() or 1.0
    full_df.loc[mask, col] = full_df.loc[mask, col] + rng.normal(8.0 * col_std, col_std, size=mask.sum())

# rebuild the cold-start pool + batches from the now-drift-injected table (group
# membership is unaffected — simulate_batches doesn't look at feature values)
initial_df, batches = simulate_batches(
    full_df, group_column=CONFIG["group_column"], id_column=CONFIG["id_column"],
    n_initial_groups=CONFIG["n_initial_groups"], batch_size_groups=CONFIG["batch_size_groups"],
    seed=CONFIG["seed"],
)
batches = batches[: CONFIG["max_batches"]]
print(f"Batch {drift_batch_idx} ({len(drift_plane_ids)} planes) carries the injected sensor-mean shift.")

Batch 0 (10 planes) carries the injected sensor-mean shift.


## 4. Model client + run directory

In [8]:
base_url, api_key, default_model = resolve_model_endpoint(
    CONFIG["use_gateway"], CONFIG["model"], "qwen3-coder:30b", "rit-qwen3-coder-30b",
    use_local=CONFIG["use_local"],
)
client = ModelClient(base_url=base_url, api_key=api_key, default_model=default_model)
_, _, verification_model = resolve_model_endpoint(
    CONFIG["use_gateway"], CONFIG["verification_model"], "gemma4:latest", "rit-gemma4-latest",
    use_local=CONFIG["use_local"],
)

run_id, run_dir = make_run_dir(None)
trace = make_tracer(run_dir / "trace.jsonl")
write_transcript = make_transcript_writer(run_dir)
print(f"run_id={run_id}  model={default_model}  verification_model={verification_model}")

run_id=run_51515aa8  model=Qwen/Qwen3-Coder-30B-A3B-Instruct  verification_model=Qwen/Qwen3-Coder-30B-A3B-Instruct


## 5. Cold start — one ordinary `run_dynamic_loop` call

Identical in shape to a Phase 8 run: `feature_engineering_done` is
pre-seeded `True` (see §2's note), so the planner walks straight to
`profiler -> split_and_check_leakage -> modeling -> verification ->
finalize -> summarize`. `finalize` establishes `ctx.accumulated_df`
from `ctx.engineered_df` — that's what unlocks the `"streaming"`
capability for every `run_dynamic_loop` call from here on.

In [9]:
batches_dir = run_dir / "batches"
batches_dir.mkdir(parents=True, exist_ok=True)
initial_csv = batches_dir / "initial.csv"
initial_df.to_csv(initial_csv, index=False)

id_columns = [c.strip() for c in CONFIG["id_columns"].split(",") if c.strip()]
metric_names = CONFIG["metrics"].split(",")
goal_text = f"predict {CONFIG['target_column']}"  # deliberately plain — see run_streaming_monitor.py's note

ctx = DynamicRunContext(
    data_path=str(initial_csv), goal=goal_text, seed=CONFIG["seed"],
    target_column=CONFIG["target_column"], group_column=CONFIG["group_column"],
    id_columns=id_columns, strategy_override=CONFIG["strategy"], metric_names=metric_names,
    run_id=run_id,
)
state = RunStateSummary(goal=goal_text)
state.target_known = True
state.target_column = CONFIG["target_column"]
state.feature_engineering_done = True
load_raw_hash(ctx)

cold_start_result = run_dynamic_loop(
    ctx, state, client, model=default_model, verification_model=verification_model,
    max_iterations=CONFIG["max_iterations_cold_start"],
    trace_fn=lambda r: trace(**r), write_transcript=write_transcript,
)
print(f"Cold-start status: {cold_start_result.status}")
print(f"Model v{ctx.model_version} ready — accumulated_df now has {len(ctx.accumulated_df)} rows.")
if ctx.final_test_metrics:
    display(pd.DataFrame(ctx.final_test_metrics).T)

/home/ubuntupc/Documents/Github/agentic-ai-project/agentic-ml-classification/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


/home/ubuntupc/Documents/Github/agentic-ai-project/agentic-ml-classification/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/ubuntupc/Documents/Github/agentic-ai-project/agentic-ml-classification/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of

Cold-start status: success
Model v1 ready — accumulated_df now has 49 rows.


,metric,value,ci_low,ci_high,n_bootstrap
roc_auc,roc_auc,0.5,0.0,1.0,200
pr_auc,pr_auc,0.542857,0.225,0.967143,199
f1,f1,0.333333,0.0,0.8,198
accuracy,accuracy,0.5,0.25,0.875,200


## 6. Replay the batches

For each batch: push it into `(ctx, state)`, call `run_dynamic_loop`
again (small `max_iterations` — a batch cycle is at most
`monitor_drift -> retrain_decision -> (infer_batch | a full retrain)`),
then print what the planner actually decided and why, straight from
`state.drift_summary`/`state.pending_retrain_action` — no
reinterpretation.

In [10]:
def push_batch(ctx, state, batch_df):
    ctx.pending_batch_df = batch_df
    state.new_batch_pending = True
    state.drift_checked = False
    state.drift_summary = None
    state.pending_retrain_action = None
    state.batch_action_completed = False


batch_results = []
for i, batch_df in enumerate(batches):
    model_version_before = ctx.model_version
    push_batch(ctx, state, batch_df)
    result = run_dynamic_loop(
        ctx, state, client, model=default_model, verification_model=verification_model,
        max_iterations=CONFIG["max_iterations_per_batch"],
        trace_fn=lambda r: trace(**r), write_transcript=write_transcript,
    )
    retrained = ctx.model_version != model_version_before
    batch_results.append({
        "batch_index": i, "n_examples": len(batch_df), "status": result.status,
        "drift_summary": state.drift_summary, "decision": state.pending_retrain_action,
        "model_version_before": model_version_before, "model_version_after": ctx.model_version,
        "retrained": retrained,
    })

    tag = "DRIFT-INJECTED" if i == drift_batch_idx else "normal"
    mean_shift = (state.drift_summary or {}).get("mean_abs_shift")
    display(Markdown(
        f"**Batch {i}** ({tag}, {len(batch_df)} rows) — status=`{result.status}`, "
        f"mean_abs_shift={mean_shift}, decision=**{state.pending_retrain_action}**, "
        f"model v{model_version_before} -> v{ctx.model_version}"
    ))
    ctx.pending_batch_df = None

/home/ubuntupc/Documents/Github/agentic-ai-project/agentic-ml-classification/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


/home/ubuntupc/Documents/Github/agentic-ai-project/agentic-ml-classification/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/ubuntupc/Documents/Github/agentic-ai-project/agentic-ml-classification/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of

**Batch 0** (DRIFT-INJECTED, 20 rows) — status=`max_iterations_reached`, mean_abs_shift=10.085836, decision=**retrain**, model v1 -> v1

**Batch 1** (normal, 16 rows) — status=`success`, mean_abs_shift=0.62428, decision=**infer_only**, model v1 -> v1

**On batch 0's `max_iterations_reached`:** the drift detection and
retrain *decision* are both correct here (`mean_abs_shift=10.09`, far
past the `2.0` threshold, correctly triggers `"retrain"`) — what didn't
finish in time was the retrain's own `modeling` step, which kept
proposing a candidate config referencing `"split"` (one of this run's
`--id-columns`, already dropped from the feature matrix) and never
self-corrected from the resulting error within this batch's
20-iteration budget. That's a real, pre-existing weakness of the local
30B model's `modeling` agent on this dataset, unrelated to anything
Phase 9 adds — the routing mechanism (`monitor_drift` -> a correct
`"retrain"` decision -> the classification-phase reset) all worked
exactly as designed. `runs/streaming_eval_report.md`
(`scripts/evaluate_streaming_monitor.py`) shows the same drift-batch
routing on a separate run where the retrain's `modeling` step happened
to succeed, completing the cycle end-to-end with `model_version`
incrementing.

## 7. Model-version history

In [11]:
if ctx.model_history:
    history_df = pd.DataFrame(ctx.model_history)
    history_df["test_metrics"] = history_df["test_metrics"].apply(
        lambda m: {k: round(v["value"], 4) for k, v in m.items()} if m else None
    )
    display(history_df[["version", "n_training_examples", "test_metrics"]])
else:
    print("No model_history recorded.")

,version,n_training_examples,test_metrics
0,1,49,"{'roc_auc': 0.5, 'pr_auc': 0.5429, 'f1': 0.333..."


## 8. What this notebook did NOT need to reimplement

Every classification step in every retrain cycle above ran through the
exact same `steps/profiler_step.py` / `steps/split_step.py` /
`steps/modeling_step.py` / `steps/verification_step.py` /
`steps/finalize_step.py` functions `run_orchestrator.py` and
`run_dynamic_orchestrator.py` already use — zero new modeling logic.
`monitor_drift` (`harness/drift.py`) and the batch simulator
(`harness/streaming.py`) are the only new deterministic code;
`retrain_decision` (`steps/retrain_decision_step.py`) is the only new
agent, and it only ever proposes — `orchestrator/dynamic_loop.py`'s
`execute_agent_step` is what actually merges a batch into
`accumulated_df`, resets state for a retrain cycle, or scores a batch,
after `validate_plan` has independently re-checked every precondition.

For a scenario with real, unambiguous injected drift proven end-to-end
against this exact local model (drift correctly triggers `"retrain"`,
non-drift batches correctly get `"infer_only"`), see
`runs/streaming_eval_report.md`
(`scripts/evaluate_streaming_monitor.py`) — the real-LLM evaluation
counterpart to this notebook's walkthrough.

## 9. Artifacts written this run

In [12]:
report = {
    "run_id": run_id, "model": default_model, "cold_start_status": cold_start_result.status,
    "final_state": state.to_planner_dict(), "model_history": ctx.model_history,
    "batch_results": batch_results,
}
out_path = run_dir / "streaming_monitor_notebook_report.json"
out_path.write_text(json.dumps(report, indent=2, default=str))
print(f"Run directory: {run_dir}")
print(f"Report: {out_path}")
print(f"Transcripts: {run_dir / 'transcripts'}")

Run directory: runs/run_51515aa8
Report: runs/run_51515aa8/streaming_monitor_notebook_report.json
Transcripts: runs/run_51515aa8/transcripts
